## Data Loading & Preparation

In [6]:
import pandas as pd
import spacy
from spacy.tokens import Doc
from transformers import pipeline
from sklearn.metrics import classification_report

ner_df= pd.read_csv("NER-test.tsv", sep="\t")
st_df= pd.read_csv("Sentiment-topic-test.tsv", sep="\t")

sentences= []
for sid, grp in ner_df.groupby("sentence id"):
    grp= grp.sort_values("token id")
    meta= st_df[st_df["sentence id"] == sid].iloc[0]
    sentences.append({
        "sid": int(sid),
        "tokens": grp["token"].tolist(),
        "gold_bio": grp["BIO NER tag"].tolist(),
        "text": meta["text"],
    })
print(f"Success! {len(sentences)} sentences loaded.")

Success! 10 sentences loaded.


## spaCy

In [3]:
print("Loading spaCy model...")
nlp= spacy.load("en_core_web_sm")

SPACY2GOLD= {
    "PERSON":"PER", "ORG":"ORG",
    "GPE":"LOC", "LOC":"LOC", "FAC":"LOC",
    "NORP":"MISC", "LANGUAGE":"MISC", "WORK_OF_ART":"MISC",
    "EVENT":"MISC", "PRODUCT":"MISC", "LAW":"MISC",
} 

def spacy_bio(tokens):
    doc= Doc(nlp.vocab, words=tokens)        
    for _, pipe in nlp.pipeline:
        doc= pipe(doc)
    out= []
    for tok in doc:
        if tok.ent_iob_ == "O":
            out.append("O")
        else:
            m= SPACY2GOLD.get(tok.ent_type_)
            out.append(f"{tok.ent_iob_}-{m}" if m else "O")
    return out

print("Generating predictions with spaCy...")
spacy_predictions= [spacy_bio(s["tokens"]) for s in sentences]
print("spaCy predictions complete!")

Loading spaCy model...
Generating predictions with spaCy...
spaCy predictions complete!


: 

## BERT

In [4]:
bert_ner= pipeline("ner", model="dslim/bert-base-NER")

def bert_bio(text, tokens):
    preds= bert_ner(text)
    out= ["O"] * len(tokens)
    
    current_idx= 0
    token_spans= []
    for token in tokens:
        start= text.find(token, current_idx)
        end= start + len(token)
        token_spans.append((start, end))
        current_idx= end
        
    for p in preds:
        p_start, p_end, p_ent = p['start'], p['end'], p['entity']
        for i, (t_start, t_end) in enumerate(token_spans):
            if max(p_start, t_start) < min(p_end, t_end): 
                if out[i] == "O":
                    out[i]= p_ent
    return out

bert_predictions = [bert_bio(s["text"], s["tokens"]) for s in sentences]

Loading Hugging Face BERT model (this may take a minute)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 13482.59it/s]
[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generating predictions with BERT...
BERT predictions complete!


: 

## Evaluation Metrics

In [5]:
gold_flat= []
spacy_flat= []
bert_flat= []

for idx, s in enumerate(sentences):
    gold_flat.extend(s["gold_bio"])
    spacy_flat.extend(spacy_predictions[idx])
    bert_flat.extend(bert_predictions[idx])

print("          SYSTEM 1: SPACY PERFORMANCE")
print(classification_report(gold_flat, spacy_flat, zero_division=0))


print("          SYSTEM 2: BERT PERFORMANCE")
print(classification_report(gold_flat, bert_flat, zero_division=0))


          SYSTEM 1: SPACY PERFORMANCE
              precision    recall  f1-score   support

       B-LOC       1.00      1.00      1.00         4
      B-MISC       0.75      1.00      0.86         3
       B-ORG       0.67      0.50      0.57         4
       B-PER       0.67      0.67      0.67         6
       I-LOC       1.00      1.00      1.00         2
      I-MISC       1.00      1.00      1.00         1
       I-ORG       0.75      1.00      0.86         3
       I-PER       1.00      0.75      0.86         8
           O       0.99      0.99      0.99       183

    accuracy                           0.97       214
   macro avg       0.87      0.88      0.87       214
weighted avg       0.97      0.97      0.97       214


          SYSTEM 2: BERT PERFORMANCE
              precision    recall  f1-score   support

       B-LOC       0.80      1.00      0.89         4
      B-MISC       1.00      1.00      1.00         3
       B-ORG       1.00      0.75      0.86         4
 

: 